In [9]:
#!/usr/bin/env python
# coding=utf-8
"""
VQL SQL Matcher

Function: Compare whether the SQL execution results in two VQLs match
"""

import os
import pandas as pd
import re
import sqlite3
import tempfile
from typing import Optional, Tuple
import json
from tqdm import tqdm
import numpy as np



class VQLSQLMatcher:
    """VQL SQL Matcher"""
    
    def __init__(self, database_path):
        self.database_path = database_path
        self.temp_db_path = None
    
    def match_vql_results(self, vql_predicted: str, vql_gt: str, db_id: str) -> Tuple[bool, Optional[str]]:
        """
        Compare whether the execution results of two VQLs match

        Args:
            vql_predicted: The predicted VQL
            vql_gt: The ground truth VQL
            db_id: Database ID

        Returns:
            (Whether they match, error message)
        """
        try:
            # 1. Extract and compare Visualize part first
            viz_predicted = self._extract_visualize_type(vql_predicted)
            viz_gt = self._extract_visualize_type(vql_gt)

            # If Visualize types don't match, return False immediately
            if viz_predicted != viz_gt:
                return False, f"Visualize types don't match: predicted='{viz_predicted}', ground_truth='{viz_gt}'"

            # 2. Create temporary database
            self._create_temp_database(db_id)

            # 3. Extract SQL part
            sql_predicted = self._extract_sql_from_vql(vql_predicted)
            sql_gt = self._extract_sql_from_vql(vql_gt)

            if not sql_predicted or not sql_gt:
                if not sql_predicted:
                    return False, "Failed to extract predicted SQL part"
                if not sql_gt:
                    return False, "Failed to extract ground truth SQL part"

            # 4. Execute SQL queries
            result_gt = self._execute_sql(sql_gt, "gt")
            result_predicted = self._execute_sql(sql_predicted, "predicted")

            if result_gt is None:
                return False, "Ground truth SQL execution failed"
            elif result_predicted is None:
                return False, "Predicted SQL execution failed"

            # 5. Compare results
            is_match = self._compare_results(result_predicted, result_gt)

            # if not is_match:
            #     print(f"SQL same but results different:")
            #     print(f"SQL_predicted: {sql_predicted}")
            #     print(f"SQL_gt: {sql_gt}")
            #     print(f"Result_predicted: {result_predicted}")
            #     print(f"Result_gt: {result_gt}")
            #     exit()
            return is_match, None

        except Exception as e:
            print(f"Error: {e}")
            exit()
        finally:
            self._cleanup_temp_database()
    
    def _extract_visualize_type(self, vql: str) -> Optional[str]:
        """Extract visualize type from VQL"""
        if not vql:
            return None

        # Match Visualize keyword followed by chart type
        viz_match = re.search(r'Visualize\s+(\w+)', vql, re.IGNORECASE)
        if viz_match:
            return viz_match.group(1).lower()  # Normalize to lowercase
        return None

    def _extract_sql_from_vql(self, vql: str) -> Optional[str]:
        """Extract and clean SQL from VQL - remove Visualize / BIN clauses"""
        if not vql:
            return None

        # 1. Remove Visualize prefix, keep only SQL body
        sql_match = re.search(r'(?:Visualize\s+\w+\s+)?(SELECT.*?)(?:\s*$)',
                            vql, re.IGNORECASE | re.DOTALL)
        if not sql_match:
            return None
        sql = sql_match.group(1).strip()

        bin_pattern = re.compile(
            r'(?:,?\s*BIN\s+\S+(?:\s+BY\s+\S+)?\s*)',
            re.IGNORECASE
        )
        sql = re.sub(bin_pattern, ' ', sql)

        # 3. Clean up extra spaces
        sql = re.sub(r'\s+', ' ', sql).strip()
        return sql


    
    def _create_temp_database(self, db_id: str):
        """Create temporary SQLite database"""
        db_path = os.path.join(self.database_path, db_id)
        
        # Create temporary database file
        temp_fd, self.temp_db_path = tempfile.mkstemp(suffix='.db')
        os.close(temp_fd)

        # Connect to temporary database
        conn = sqlite3.connect(self.temp_db_path)

        try:
            # Read CSV files and create tables
            for file_name in os.listdir(db_path):
                if file_name.endswith('.csv'):
                    table_name = file_name[:-4]  # Remove .csv suffix

                    # Skip SQLite system table names
                    if table_name.lower() in ['sqlite_sequence']:
                        continue
                    
                    file_path = os.path.join(db_path, file_name)
                    
                    # Read CSV file
                    df = pd.read_csv(file_path)

                    # Write to SQLite database
                    df.to_sql(table_name, conn, if_exists='replace', index=False)
                    
        finally:
            conn.close()
    
    def _execute_sql(self, sql: str, type: str) -> Optional[pd.DataFrame]:
        """Execute SQL query"""
        if not sql or not self.temp_db_path:
            return None
        
        conn = None
        try:
            conn = sqlite3.connect(self.temp_db_path)
            result = pd.read_sql_query(sql, conn)
            return result
        except Exception as e:
            # print(f"{type} SQL execution error: {e}")
            # print(f"{type} SQL: {sql}")
            return None
        finally:
            # Ensure connection is closed in finally block
            if conn:
                conn.close()
    
    def _compare_results(self, result1: pd.DataFrame, result2: pd.DataFrame) -> bool:
        """Compare whether two query results match"""
        # If column count or row count differ, they don't match
        if result1.shape != result2.shape:
            return False

        # Reset index for comparison
        result1 = result1.reset_index(drop=True)
        result2 = result2.reset_index(drop=True)

        # Ignore column name differences, only compare data content
        # Reset column names to numeric indices
        result1_no_cols = result1.copy()
        result2_no_cols = result2.copy()
        result1_no_cols.columns = range(len(result1_no_cols.columns))
        result2_no_cols.columns = range(len(result2_no_cols.columns))

        # Use pandas equals method for comparison, which handles NaN values correctly
        return result1_no_cols.equals(result2_no_cols)
    
    def _cleanup_temp_database(self):
        """Clean up temporary database"""
        if self.temp_db_path and os.path.exists(self.temp_db_path):
            import time
            import gc
            
            # Force garbage collection to ensure connections are fully released
            gc.collect()

            # Try multiple times to delete, with delay
            max_retries = 3
            for i in range(max_retries):
                try:
                    os.remove(self.temp_db_path)
                    break  # Successfully deleted, exit loop
                except PermissionError as e:
                    if i < max_retries - 1:  # If not the last attempt
                        time.sleep(0.1)  # Wait 100ms then retry
                        continue
                    else:
                        # Last attempt failed, only warn but don't exit program
                        print(f"Warning: Failed to delete temporary database file {self.temp_db_path}: {e}")
                except Exception as e:
                    # Other exceptions, only warn but don't exit program
                    print(f"Warning: Exception occurred while cleaning up temporary database: {e}")
                    break


def match_vql_pair(vql_predicted: str, vql_gt: str, db_id: str, database_path: str) -> Tuple[bool, Optional[str]]:
    """
    Convenience function: Match two VQLs

    Args:
        vql_predicted: The predicted VQL
        vql_gt: The ground truth VQL
        db_id: Database ID

    Returns:
        (Whether they match, error message)
    """
    matcher = VQLSQLMatcher(database_path)
    return matcher.match_vql_results(vql_predicted, vql_gt, db_id)


def main(result_path, database_path):

    # Load main results
    result = json.load(open(result_path, 'r'))

    print("Total data size:", len(result))

    # Calculate overall accuracy
    correct_count = 0
    incorrect_count = 0
    gt_sql_err_exec_count, predicted_sql_err_exec_count, gt_sql_err_extract_count, predicted_sql_err_extract_count, total_count, dismatch_count = 0, 0, 0, 0, 0, 0

    print(f"\nProcessing overall dataset ({len(result)} items)...")

    for item in tqdm(result, desc="Overall Processing"):
        vql_predicted = item['final_dvq']
        vql_gt = item['target']
        is_match, error = match_vql_pair(vql_predicted, vql_gt, item['db_id'], database_path)

        if is_match:
            correct_count += 1
        else:
            incorrect_count += 1
            if error == "Ground truth SQL execution failed":
                gt_sql_err_exec_count += 1
            elif error == "Predicted SQL execution failed":
                predicted_sql_err_exec_count += 1
            elif error == "Failed to extract predicted SQL part":
                predicted_sql_err_extract_count += 1
            elif error == "Failed to extract ground truth SQL part":
                gt_sql_err_extract_count += 1
            elif error == None or error.startswith("Visualize types don't match"):
                dismatch_count += 1
        total_count += 1

    overall_accuracy = correct_count / (correct_count + incorrect_count) if (correct_count + incorrect_count) > 0 else 0

    # print("Overall results:")
    # print(f"Data size: {total_count}")
    # print(f"Correct count: {correct_count}")
    # print(f"Incorrect count: {incorrect_count}")
    # print(f"Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")
    # print(f"GT SQL extraction errors: {gt_sql_err_extract_count}")
    # print(f"Predicted SQL extraction errors: {predicted_sql_err_extract_count}")
    # print(f"GT SQL execution errors: {gt_sql_err_exec_count}")
    # print(f"Predicted SQL execution errors: {predicted_sql_err_exec_count}")
    # print(f"Dismatch count: {dismatch_count}")
    # print(f"\n{'='*50}")

    return overall_accuracy



if __name__ == "__main__":
    
    # result_path = f"./nvBench-Rob/dev_nlq/result_multi-turn/rebuttal/dev_nlq_result_multi-turn_gpt-3.5-turbo_0.json"
    database_path = "./nvBench-Rob/database_csv"
    # main(result_path, database_path) 

    for mode in ['dev_nlq', 'dev_schema', 'dev_nlq_schema']:
        overall_accuracy_list = []
        for i in range(5):
            result_path = f"./nvBench-Rob/{mode}/result_multi-turn/rebuttal/{mode}_result_multi-turn_qwen_{i}.json"
            overall_accuracy = main(result_path, database_path)
            print(f"overall_accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")
            overall_accuracy_list.append(overall_accuracy)
        print(f"===============================mode: {mode}===============================")
        print(f"overall_accuracy_list: {overall_accuracy_list}")
        print(f"avg_overall_accuracy: {sum(overall_accuracy_list) / len(overall_accuracy_list):.4f}")
        print(f"std_overall_accuracy: {np.std(overall_accuracy_list):.4f}")

Total data size: 1182

Processing overall dataset (1182 items)...


Overall Processing: 100%|██████████| 1182/1182 [02:20<00:00,  8.44it/s]


overall_accuracy: 0.8020 (80.20%)
Total data size: 1182

Processing overall dataset (1182 items)...


Overall Processing: 100%|██████████| 1182/1182 [02:19<00:00,  8.47it/s]


overall_accuracy: 0.8071 (80.71%)
Total data size: 1182

Processing overall dataset (1182 items)...


...

overall_accuracy: 0.7477 (74.77%)
Total data size: 1185

Processing overall dataset (1185 items)...


Overall Processing: 100%|██████████| 1185/1185 [02:28<00:00,  7.97it/s]


overall_accuracy: 0.7502 (75.02%)
Total data size: 1185

Processing overall dataset (1185 items)...


Overall Processing: 100%|██████████| 1185/1185 [03:18<00:00,  5.97it/s]

overall_accuracy: 0.7511 (75.11%)
===============================mode: dev_nlq_schema===============================
overall_accuracy_list: [0.7561181434599156, 0.7468354430379747, 0.7476793248945147, 0.750210970464135, 0.7510548523206751]
avg_overall_accuracy: 0.7504
std_overall_accuracy: 0.0033


In [13]:
#!/usr/bin/env python
# coding=utf-8
"""
VQL SQL Matcher

Function: Compare whether the SQL execution results in two VQLs match
"""

import os
import pandas as pd
import re
import sqlite3
import tempfile
from typing import Optional, Tuple
import json
from tqdm import tqdm
import numpy as np



class VQLSQLMatcher:
    """VQL SQL Matcher"""
    
    def __init__(self, database_path):
        self.database_path = database_path
        self.temp_db_path = None
    
    def match_vql_results(self, vql_predicted: str, vql_gt: str, db_id: str) -> Tuple[bool, Optional[str]]:
        """
        Compare whether the execution results of two VQLs match

        Args:
            vql_predicted: The predicted VQL
            vql_gt: The ground truth VQL
            db_id: Database ID

        Returns:
            (Whether they match, error message)
        """
        try:
            # 1. Extract and compare Visualize part first
            viz_predicted = self._extract_visualize_type(vql_predicted)
            viz_gt = self._extract_visualize_type(vql_gt)

            # If Visualize types don't match, return False immediately
            if viz_predicted != viz_gt:
                return False, f"Visualize types don't match: predicted='{viz_predicted}', ground_truth='{viz_gt}'"

            # 2. Create temporary database
            self._create_temp_database(db_id)

            # 3. Extract SQL part
            sql_predicted = self._extract_sql_from_vql(vql_predicted)
            sql_gt = self._extract_sql_from_vql(vql_gt)

            if not sql_predicted or not sql_gt:
                if not sql_predicted:
                    return False, "Failed to extract predicted SQL part"
                if not sql_gt:
                    return False, "Failed to extract ground truth SQL part"

            # 4. Execute SQL queries
            result_gt = self._execute_sql(sql_gt, "gt")
            result_predicted = self._execute_sql(sql_predicted, "predicted")

            if result_gt is None:
                return False, "Ground truth SQL execution failed"
            elif result_predicted is None:
                return False, "Predicted SQL execution failed"

            # 5. Compare results
            is_match = self._compare_results(result_predicted, result_gt)

            # if not is_match:
            #     print(f"SQL same but results different:")
            #     print(f"SQL_predicted: {sql_predicted}")
            #     print(f"SQL_gt: {sql_gt}")
            #     print(f"Result_predicted: {result_predicted}")
            #     print(f"Result_gt: {result_gt}")
            #     exit()
            return is_match, None

        except Exception as e:
            print(f"Error: {e}")
            exit()
        finally:
            self._cleanup_temp_database()
    
    def _extract_visualize_type(self, vql: str) -> Optional[str]:
        """Extract visualize type from VQL"""
        if not vql:
            return None

        # Match Visualize keyword followed by chart type
        viz_match = re.search(r'Visualize\s+(\w+)', vql, re.IGNORECASE)
        if viz_match:
            return viz_match.group(1).lower()  # Normalize to lowercase
        return None

    def _extract_sql_from_vql(self, vql: str) -> Optional[str]:
        """Extract and clean SQL from VQL - remove Visualize / BIN clauses"""
        if not vql:
            return None

        # 1. Remove Visualize prefix, keep only SQL body
        sql_match = re.search(r'(?:Visualize\s+\w+\s+)?(SELECT.*?)(?:\s*$)',
                            vql, re.IGNORECASE | re.DOTALL)
        if not sql_match:
            return None
        sql = sql_match.group(1).strip()

        bin_pattern = re.compile(
            r'(?:,?\s*BIN\s+\S+(?:\s+BY\s+\S+)?\s*)',
            re.IGNORECASE
        )
        sql = re.sub(bin_pattern, ' ', sql)

        # 3. Clean up extra spaces
        sql = re.sub(r'\s+', ' ', sql).strip()
        return sql


    
    def _create_temp_database(self, db_id: str):
        """Create temporary SQLite database"""
        db_path = os.path.join(self.database_path, db_id)
        
        # Create temporary database file
        temp_fd, self.temp_db_path = tempfile.mkstemp(suffix='.db')
        os.close(temp_fd)

        # Connect to temporary database
        conn = sqlite3.connect(self.temp_db_path)

        try:
            # Read CSV files and create tables
            for file_name in os.listdir(db_path):
                if file_name.endswith('.csv'):
                    table_name = file_name[:-4]  # Remove .csv suffix

                    # Skip SQLite system table names
                    if table_name.lower() in ['sqlite_sequence']:
                        continue
                    
                    file_path = os.path.join(db_path, file_name)
                    
                    # Read CSV file
                    df = pd.read_csv(file_path)

                    # Write to SQLite database
                    df.to_sql(table_name, conn, if_exists='replace', index=False)
                    
        finally:
            conn.close()
    
    def _execute_sql(self, sql: str, type: str) -> Optional[pd.DataFrame]:
        """Execute SQL query"""
        if not sql or not self.temp_db_path:
            return None
        
        conn = None
        try:
            conn = sqlite3.connect(self.temp_db_path)
            result = pd.read_sql_query(sql, conn)
            return result
        except Exception as e:
            # print(f"{type} SQL execution error: {e}")
            # print(f"{type} SQL: {sql}")
            return None
        finally:
            # Ensure connection is closed in finally block
            if conn:
                conn.close()
    
    def _compare_results(self, result1: pd.DataFrame, result2: pd.DataFrame) -> bool:
        """Compare whether two query results match"""
        # If column count or row count differ, they don't match
        if result1.shape != result2.shape:
            return False

        # Reset index for comparison
        result1 = result1.reset_index(drop=True)
        result2 = result2.reset_index(drop=True)

        # Ignore column name differences, only compare data content
        # Reset column names to numeric indices
        result1_no_cols = result1.copy()
        result2_no_cols = result2.copy()
        result1_no_cols.columns = range(len(result1_no_cols.columns))
        result2_no_cols.columns = range(len(result2_no_cols.columns))

        # Use pandas equals method for comparison, which handles NaN values correctly
        return result1_no_cols.equals(result2_no_cols)
    
    def _cleanup_temp_database(self):
        """Clean up temporary database"""
        if self.temp_db_path and os.path.exists(self.temp_db_path):
            import time
            import gc
            
            # Force garbage collection to ensure connections are fully released
            gc.collect()

            # Try multiple times to delete, with delay
            max_retries = 3
            for i in range(max_retries):
                try:
                    os.remove(self.temp_db_path)
                    break  # Successfully deleted, exit loop
                except PermissionError as e:
                    if i < max_retries - 1:  # If not the last attempt
                        time.sleep(0.1)  # Wait 100ms then retry
                        continue
                    else:
                        # Last attempt failed, only warn but don't exit program
                        print(f"Warning: Failed to delete temporary database file {self.temp_db_path}: {e}")
                except Exception as e:
                    # Other exceptions, only warn but don't exit program
                    print(f"Warning: Exception occurred while cleaning up temporary database: {e}")
                    break


def match_vql_pair(vql_predicted: str, vql_gt: str, db_id: str, database_path: str) -> Tuple[bool, Optional[str]]:
    """
    Convenience function: Match two VQLs

    Args:
        vql_predicted: The predicted VQL
        vql_gt: The ground truth VQL
        db_id: Database ID

    Returns:
        (Whether they match, error message)
    """
    matcher = VQLSQLMatcher(database_path)
    return matcher.match_vql_results(vql_predicted, vql_gt, db_id)


def main(result_path, database_path):

    # Load main results
    result = json.load(open(result_path, 'r', encoding='gbk'))

    print("Total data size:", len(result))

    # Calculate overall accuracy
    correct_count = 0
    incorrect_count = 0
    gt_sql_err_exec_count, predicted_sql_err_exec_count, gt_sql_err_extract_count, predicted_sql_err_extract_count, total_count, dismatch_count = 0, 0, 0, 0, 0, 0

    print(f"\nProcessing overall dataset ({len(result)} items)...")

    c=0
    for item in tqdm(result, desc="Overall Processing"):
        vql_predicted = item['predict_rag_nlq']
        vql_gt = item['target']
        is_match, error = match_vql_pair(vql_predicted, vql_gt, item['db_id'], database_path)

        if is_match:
            correct_count += 1
        else:    
            if error == "Ground truth SQL execution failed":
                gt_sql_err_exec_count += 1
                # continue
            elif error == "Predicted SQL execution failed":
                predicted_sql_err_exec_count += 1
            elif error == "Failed to extract predicted SQL part":
                predicted_sql_err_extract_count += 1
            elif error == "Failed to extract ground truth SQL part":
                gt_sql_err_extract_count += 1
            elif error == None or error.startswith("Visualize types don't match"):
                dismatch_count += 1
            incorrect_count += 1
        total_count += 1

    overall_accuracy = correct_count / (correct_count + incorrect_count) if (correct_count + incorrect_count) > 0 else 0

    # print("Overall results:")
    # print(f"Data size: {total_count}")
    # print(f"Correct count: {correct_count}")
    # print(f"Incorrect count: {incorrect_count}")
    # print(f"Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")
    # print(f"GT SQL extraction errors: {gt_sql_err_extract_count}")
    # print(f"Predicted SQL extraction errors: {predicted_sql_err_extract_count}")
    # print(f"GT SQL execution errors: {gt_sql_err_exec_count}")
    # print(f"Predicted SQL execution errors: {predicted_sql_err_exec_count}")
    # print(f"Dismatch count: {dismatch_count}")
    # print(f"\n{'='*50}")

    return overall_accuracy



if __name__ == "__main__":

    for mode in ['dev_nlq', 'dev_schema', 'dev_nlq_schema']:
    # for mode in ['dev_nlq_schema']:
        print(f"===============================mode: {mode}===============================")
        # result_path = f"./self-debug/{mode}/{mode}_exec_debug.json"
        result_path = f"./nvBench-Rob/{mode}/result_rebuttal/{mode}_result_few_shot_gpt3.5.json"
        database_path = "./nvBench-Rob/database_csv"
        overall_accuracy = main(result_path, database_path) 
        print(f"overall_accuracy: {overall_accuracy}")
        print(f"===============================mode: {mode}===============================")



===============================mode: dev_nlq===============================
Total data size: 1182

Processing overall dataset (1182 items)...


Overall Processing: 100%|██████████| 1182/1182 [02:23<00:00,  8.24it/s]


overall_accuracy: 0.40016920473773265
===============================mode: dev_nlq===============================
===============================mode: dev_schema===============================
Total data size: 1182

Processing overall dataset (1182 items)...


Overall Processing: 100%|██████████| 1182/1182 [02:35<00:00,  7.62it/s]


overall_accuracy: 0.36209813874788493
===============================mode: dev_schema===============================
===============================mode: dev_nlq_schema===============================
Total data size: 1182

Processing overall dataset (1182 items)...


Overall Processing: 100%|██████████| 1182/1182 [02:45<00:00,  7.16it/s]

overall_accuracy: 0.338409475465313
===============================mode: dev_nlq_schema===============================
